# Project Description
By leveraging AI, unstructured natural language can be processed into organized data. This can reduce administrative tasks for healthcare professionals, freeing more time for direct patient care.

In this project, you'll help the medical team automate the extraction and interpretation of vital information from their transcripts using the OpenAI API.

# Project Instructions

You have been provided with an anonymized dataset of medical transcriptions organized by specialty, transcriptions.csv.

Use the OpenAI API to extract "age", "medical_specialty", and a new data field to store the recommended treatment extracted from each transcription.
Match each recommended treatment with the corresponding International Classification of Diseases (ICD) code, and save your answers in a pandas DataFrame named df_structured.
*Note that when clicking "Submit Project" it may take a while for the code to execute and the tests to run as you are interacting with the OpenAI API.

![electronic_medical_records](electronic_medical_records.png)

Medical professionals often summarize patient encounters in transcripts written in natural language, which include details about symptoms, diagnosis, and treatments. These transcripts can be used for other medical documentation, such as for insurance purposes, but as they are densely packed with medical information, extracting the key data accurately can be challenging.  

You and your team at Lakeside Healthcare Network have decided to leverage the OpenAI API to automatically extract medical information from these transcripts and automate the matching with the appropriate ICD-10 codes. ICD-10 codes are a standardized system used worldwide for diagnosing and billing purposes, such as insurance claims processing.

## The Data
The dataset contains anonymized medical transcriptions categorized by specialty.

## transcriptions.csv
| Column     | Description              |
|------------|--------------------------|
| `"medical_specialty"` | The medical specialty associated with each transcription.  |
| `"transcription"` | Detailed medical transcription texts, with insights into the medical case. |

In [ ]:
# Import the necessary libraries
import pandas as pd
from openai import OpenAI
import json

In [ ]:
# Load the data
df = pd.read_csv("data/transcriptions.csv")
df.head()

In [ ]:
# Initialize the OpenAI client
client = OpenAI()

## Start coding here, use as many cells as you need

# Define the system prompt to instruct the model on what to extract and how to format it
system_prompt = """
You are an expert medical data extractor. Read the medical transcription and extract the following information:
1. "age": The patient's age.
2. "medical_specialty": The relevant medical specialty.
3. "recommended_treatment": The recommended treatment plan or medication.
4. "icd_code": The corresponding ICD-10 code for the recommended treatment.

Respond ONLY with a valid JSON object using exactly these keys: "age", "medical_specialty", "recommended_treatment", and "icd_code".
"""

# Create an empty list to store the extracted dictionaries
extracted_records = []

# Iterate over each row in the transcriptions DataFrame
for index, row in df.iterrows():
    transcription_text = row['transcription']

    # Call the OpenAI API
    response = client.chat.completions.create(
        model="gpt-3.5-turbo", # You can use gpt-4 if the environment requires/allows it
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": transcription_text}
        ],
        response_format={"type": "json_object"}, # Forces the API to return a valid JSON object
        temperature=0.0 # Set to 0 for highly deterministic, factual extraction
    )

    # Extract the string response and parse it into a Python dictionary
    response_content = response.choices[0].message.content
    extracted_data = json.loads(response_content)

    # Append the dictionary to our list
    extracted_records.append(extracted_data)

# Convert the list of parsed JSON objects into the requested pandas DataFrame
df_structured = pd.DataFrame(extracted_records)

# View the final structured DataFrame
df_structured.head()